# AIE464 — Final Benchmark (Colab T4)
โหมดเริ่มต้นคือ **Smoke Test**: 1 optimizer update ต่อโมเดลและ Validation 2 หน้าเต็ม ไม่มีการประเมิน Test Set

4 การทดลอง: CNN + L1, DnCNN-lite, U-Net + L1, U-Net + L1 + 0.1 × Sobel Loss

**Full Training และ Test ต้องได้รับอนุมัติแยกก่อนเปลี่ยน flags** การรันโค้ดได้ไม่แปลว่าโมเดลให้ภาพดีขึ้น

เลือก Runtime → Change runtime type → T4 GPU อัปโหลด `benchmark_bundle.zip` ครั้งเดียวไป `MyDrive/AIE464/` ไม่ต้องอัปโหลดภาพทีละไฟล์ ดูวิธีสร้าง ZIP ใน `docs/FINAL_BENCHMARK.md`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
from pathlib import Path
import sys, shutil, zipfile

DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/AIE464/mini_project')
BUNDLE_PATH = Path('/content/drive/MyDrive/AIE464/benchmark_bundle.zip')
LOCAL_ROOT = Path('/content/aie464_benchmark')
PROJECT_DIR = LOCAL_ROOT / 'mini_project'
USE_EXISTING_LOCAL = False  # True only to reuse this runtime's extracted files.

if not USE_EXISTING_LOCAL:
    if LOCAL_ROOT.exists():
        raise FileExistsError('Local extraction exists; set USE_EXISTING_LOCAL=True or restart runtime.')
    local_zip = Path('/content/benchmark_bundle.zip')
    shutil.copyfile(BUNDLE_PATH, local_zip)  # Copy your own Drive ZIP; no dataset download.
    with zipfile.ZipFile(local_zip) as bundle:
        names = bundle.namelist()
        if len(names) != len(set(names)) or bundle.testzip() is not None:
            raise ValueError('ZIP duplicates or integrity failure')
        for name in names:
            target = (LOCAL_ROOT / name).resolve()
            if not target.is_relative_to(LOCAL_ROOT.resolve()) or not name.startswith('mini_project/'):
                raise ValueError('Unsafe archive path')
        bundle.extractall(LOCAL_ROOT)
if not (PROJECT_DIR / 'src/train.py').is_file():
    raise FileNotFoundError('Benchmark source missing')
sys.path.insert(0, str(PROJECT_DIR / 'src'))

import torch, numpy, PIL, matplotlib  # Preinstalled Colab dependencies; no pip install.
from train import TrainConfig, train_experiment
from benchmark import EXPERIMENTS, load_manifest
if not torch.cuda.is_available():
    raise RuntimeError('Choose a T4 GPU runtime before continuing')
print('GPU:', torch.cuda.get_device_name())
print('Frozen splits:', load_manifest(PROJECT_DIR)['counts'])


## การตั้งค่าและด่านอนุมัติ
คง mode=`smoke` จนกว่าจะตรวจผล pilot บน T4 และได้รับอนุมัติ Training เต็ม

เมื่ออนุมัติแล้ว เปลี่ยน `RUN_FULL_TRAINING=True`, `TRAINING_APPROVED=True`, `EPOCHS=20` และตั้ง `RUN_ID` ใหม่ ใช้ค่าเดียวกันทั้ง 4 โมเดล

Resume: ใช้ `RUN_ID`, config, source, data และ runtime versions เดิม แล้วตั้ง `RESUME=True` จะกลับมาจาก epoch ที่เสร็จล่าสุด หากหยุดกลาง epoch จะรัน epoch นั้นใหม่ ไม่เพิ่มงบ updates ของผลที่บันทึกไว้


In [ ]:
RUN_FULL_TRAINING = False
TRAINING_APPROVED = False  # Set only after separate user approval.
RUN_FINAL_TEST = False
FINAL_TEST_APPROVED = False  # Independent approval; never use Test for tuning.
RESUME = False
RUN_ID = 'smoke_20261013_01'  # New ID for each new run; existing runs are protected.
EPOCHS = 1

if RUN_FULL_TRAINING and not TRAINING_APPROVED:
    raise PermissionError('Full Training not approved')
if not RUN_FULL_TRAINING and EPOCHS > 2:
    raise ValueError('Smoke is capped at 2 updates per model')
if Path(RUN_ID).name != RUN_ID or RUN_ID in ('', '.', '..'):
    raise ValueError('RUN_ID must be one folder name')
MODE = 'train' if RUN_FULL_TRAINING else 'smoke'
RUN_DIR = DRIVE_PROJECT_DIR / 'outputs/final_benchmark' / RUN_ID
print('Mode:', MODE, '| Results on Drive:', RUN_DIR)


In [ ]:
summaries = {}
for experiment in EXPERIMENTS:
    config = TrainConfig(experiment=experiment, mode=MODE, epochs=EPOCHS)
    folder = RUN_DIR / experiment
    summaries[experiment] = train_experiment(
        PROJECT_DIR, folder, config,
        resume=RESUME and folder.exists(),
        approved_training=TRAINING_APPROVED,
        require_cuda=True,
    )
    summary = summaries[experiment]
    print(experiment, '| best epoch:', summary['selected_epoch'],
          '| Val:', summary['validation']['mean'],
          '| GPU memory:', summary['gpu_memory'])


In [ ]:
from IPython.display import display, Image
for experiment in EXPERIMENTS:
    print(experiment, '| Noisy / Denoised / Clean — validation only')
    display(Image(filename=str(RUN_DIR / experiment / 'comparisons.png')))
    display(Image(filename=str(RUN_DIR / experiment / 'loss.png')))


## Final Test — ปิดไว้จนกว่าจะได้รับอนุมัติ
ต้องมี 4 Training runs ที่เสร็จครบ ใช้งบและ protocol เดียวกัน จากนั้นล็อก config และเลือก best checkpoint ด้วย Validation PSNR ก่อนตั้ง flags ทั้งสองเป็น True

Test มี 24 คู่จาก 6 กลุ่มหน้าเอกสาร มิใช่ Kaggle public `test.zip` ทุกโมเดลวัดบนหน้าเต็มชุดเดียวกัน รายงาน PSNR/SSIM, baseline ของภาพ Noisy, ภาพและ crop ตำแหน่งเดียวกัน, เวลา Inference และ GPU memory

Latency ใช้ batch 1, AMP, warm-up 2 ครั้ง แล้ว median ของ 5 forward ต่อภาพ ซิงก์ GPU และไม่นับ disk IO/การส่ง tensor/คำนวณ metrics เก็บผลแยกใน `final_test/` ห้ามนำผลกลับไปปรับโมเดล


In [ ]:
if RUN_FINAL_TEST:
    if not FINAL_TEST_APPROVED or not RUN_FULL_TRAINING:
        raise PermissionError('Final Test requires separate approval and complete training')
    from benchmark import final_benchmark
    rows = final_benchmark(PROJECT_DIR, RUN_DIR, approved=FINAL_TEST_APPROVED)
    for row in rows:
        print(row)
else:
    print('Final Test is disabled. No test images were evaluated.')
